In [5]:
# ============ 标准库导入 ============
import os
import gc
from pathlib import Path
from typing import List
from dataclasses import dataclass

# ============ 第三方库导入 ============
import polars as pl
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ============ 本地模块导入 ============
import kaggle_evaluation.default_inference_server

# ============ 全局配置 ============
# DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')
DATA_PATH: Path = Path('./data')  # 本地测试解开此行

# ============ 信号转换配置 ============
MIN_SIGNAL: float = 0.0
MAX_SIGNAL: float = 2.0
SIGNAL_MULTIPLIER: float = 400.0 

# ============ 模型配置 ============
LGBM_PARAMS = {
    "n_estimators": 5000,           # 增加树的数量，给模型更多学习空间
    "learning_rate": 0.03,          # 稍微调高学习率，避免收敛在均值
    "max_depth": 8,                 # 略微增加深度
    "num_leaves": 128,              # 增加叶子节点数
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "regression",
    "metric": "rmse",
    "n_jobs": -1,
    "random_state": 42,
    "verbose": -1,
    "min_child_samples": 10         # 降低分裂阈值，强迫模型进行学习，哪怕信号较弱
}

@dataclass
class DatasetOutput:
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    features: List[str]
    feature_means: dict  # 保存特征均值，用于推理时填充空值
    scaler: StandardScaler  # 标准化器，用于推理时标准化特征

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float
    min_signal: float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL

# ============ 1. 数据加载 ============

def load_trainset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude("date_id").cast(pl.Float64, strict=False)
        )
        .sort("date_id") 
    )

def load_testset() -> pl.DataFrame:
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude("date_id").cast(pl.Float64, strict=False)
        )
    )

# ============ 2. 特征工程 (参考 template.ipynb) ============

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.
    参考 template.ipynb 的特征提取逻辑
    用于训练时，使用 ewm_mean 填充空值

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )

def create_example_dataset_inference(df: pl.DataFrame, feature_means: dict, debug: bool = False) -> pl.DataFrame:
    """
    用于推理时的特征工程函数。
    使用训练时计算的均值来填充空值，而不是 ewm_mean（因为单行数据无法使用 ewm_mean）

    Args:
        df (pl.DataFrame): The input Polars DataFrame (通常是单行).
        feature_means (dict): 训练时计算的特征均值字典.
        debug (bool): 是否打印调试信息.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]
    
    # 基础列，需要先填充空值才能计算 U1 和 U2
    base_cols_for_u1_u2 = ["I1", "I2", "I7", "I9", "M11"]
    
    # 调试：打印原始基础列的值
    if debug:
        print("  原始基础列值:")
        for col in base_cols_for_u1_u2:
            if col in df.columns:
                val = df.select(pl.col(col)).item()
                print(f"    {col} = {val}")
    
    # 先填充基础列的空值（使用训练时的均值或0）
    fill_base_exprs = []
    for col in base_cols_for_u1_u2:
        if col in df.columns:
            mean_val = feature_means.get(col, 0.0)
            fill_base_exprs.append(pl.col(col).fill_null(mean_val))
        else:
            # 如果列不存在，用均值创建
            mean_val = feature_means.get(col, 0.0)
            fill_base_exprs.append(pl.lit(mean_val).alias(col))
    
    if fill_base_exprs:
        df = df.with_columns(fill_base_exprs)
    
    # 调试：打印填充后的基础列值
    if debug:
        print("  填充后基础列值:")
        for col in base_cols_for_u1_u2:
            val = df.select(pl.col(col)).item()
            print(f"    {col} = {val}")

    # 创建 U1 和 U2 特征（使用实际值，不填充）
    df = df.with_columns(
        (pl.col("I2") - pl.col("I1")).alias("U1"),
        (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3 + 1e-8)).alias("U2")
    )
    
    # 调试：打印计算出的 U1 和 U2
    if debug:
        u1_val = df.select(pl.col("U1")).item()
        u2_val = df.select(pl.col("U2")).item()
        print(f"  计算出的 U1 = {u1_val}, U2 = {u2_val}")
    
    # 选择需要的列
    df = df.select(["date_id", "target"] + vars_to_keep)
    
    # 使用训练时的均值填充空值（只填充原始特征列，不覆盖已计算的 U1 和 U2）
    # 注意：U1 和 U2 如果计算出来是 NaN 或 Inf，才用均值填充
    fill_exprs = []
    for col in vars_to_keep:
        if col in ["U1", "U2"]:
            # 对于 U1 和 U2，如果计算结果是 NaN 或 Inf，才用均值填充
            mean_val = feature_means.get(col, 0.0)
            fill_exprs.append(
                pl.when(pl.col(col).is_null() | pl.col(col).is_infinite())
                .then(mean_val)
                .otherwise(pl.col(col))
                .alias(col)
            )
        else:
            # 对于其他列，直接用均值填充空值
            mean_val = feature_means.get(col, 0.0)
            fill_exprs.append(pl.col(col).fill_null(mean_val))
    
    df = df.with_columns(fill_exprs)
    
    # 调试：打印最终的特征值
    if debug:
        print("  最终特征值 (前5个):")
        for i, col in enumerate(vars_to_keep[:5]):
            val = df.select(pl.col(col)).item()
            print(f"    {col} = {val}")
    
    return df
    
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_and_process_dataset(train: pl.DataFrame, test: pl.DataFrame) -> DatasetOutput:
    """
    参考 template.ipynb 的数据处理流程
    """
    # 1. 先合并训练和测试数据，统一应用特征工程
    df = join_train_test_dataframes(train, test)
    df = create_example_dataset(df=df)
    
    # 2. 分离回训练集和测试集
    train_date_ids = train.get_column('date_id').to_list()
    test_date_ids = test.get_column('date_id').to_list()
    
    train_processed = df.filter(pl.col('date_id').is_in(train_date_ids))
    test_processed = df.filter(pl.col('date_id').is_in(test_date_ids))
    
    # 3. 确定最终使用的特征列
    FEATURES: list[str] = [col for col in test_processed.columns if col not in ['date_id', 'target']]
    
    print(f"使用的特征数量: {len(FEATURES)}")
    print(f"特征列表: {FEATURES}")
    
    # 4. 计算特征均值（用于推理时填充空值）
    # 包括最终特征和用于计算 U1, U2 的基础列
    all_cols_for_means = FEATURES + ["I1", "I2", "I7", "I9", "M11"]
    feature_means = {}
    for col in all_cols_for_means:
        if col in train_processed.columns:
            mean_val = train_processed.select(pl.col(col).mean()).item()
            feature_means[col] = mean_val if not np.isnan(mean_val) else 0.0
        else:
            feature_means[col] = 0.0
    
    # 5. 转换为 Pandas 供 LGBM 使用
    X_train = train_processed.select(FEATURES).to_pandas()
    y_train = train_processed.select("target").to_pandas()["target"]
    
    X_test = test_processed.select(FEATURES).to_pandas()
    y_test = test_processed.select("target").to_pandas()["target"]
    
    # 6. 标准化特征（参考 template.ipynb）
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 转换回 DataFrame（保持特征名）
    X_train = pd.DataFrame(X_train_scaled, columns=FEATURES, index=X_train.index)
    X_test = pd.DataFrame(X_test_scaled, columns=FEATURES, index=X_test.index)
    
    return DatasetOutput(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        features=FEATURES,
        feature_means=feature_means,
        scaler=scaler
    )

# ============ 3. 模型训练 ============

def train_lgbm_model(dataset: DatasetOutput):
    X = dataset.X_train
    y = dataset.y_train
    
    tscv = TimeSeriesSplit(n_splits=5)
    models = []
    best_rmse = float('inf')
    best_model = None
    
    print(f"开始训练 LGBM...")
    
    # 交叉验证找到最佳模型
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_t, y_t = X.iloc[train_idx], y.iloc[train_idx]
        X_v, y_v = X.iloc[val_idx], y.iloc[val_idx]
        
        train_data = lgb.Dataset(X_t, label=y_t)
        val_data = lgb.Dataset(X_v, label=y_v, reference=train_data)
        
        callbacks = [
            lgb.log_evaluation(period=0),
            lgb.early_stopping(stopping_rounds=100, verbose=False) # 增加patience
        ]
        
        model = lgb.train(LGBM_PARAMS, train_data, valid_sets=[val_data], callbacks=callbacks)
        
        preds = model.predict(X_v)
        rmse = np.sqrt(mean_squared_error(y_v, preds))
        print(f"Fold {fold+1} RMSE: {rmse:.6f}")
        models.append(model)
        
        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model
    
    # 使用全量数据重新训练最终模型（使用最佳模型的迭代次数）
    print(f"使用全量数据重新训练最终模型 (最佳RMSE: {best_rmse:.6f})...")
    full_train_data = lgb.Dataset(X, label=y)
    
    # 获取最佳模型的迭代次数
    best_iteration = best_model.best_iteration if hasattr(best_model, 'best_iteration') else LGBM_PARAMS.get('n_estimators', 2000)
    
    final_model = lgb.train(
        {**LGBM_PARAMS, 'n_estimators': best_iteration},
        full_train_data,
        callbacks=[lgb.log_evaluation(period=0)]
    )
    
    # 检查模型预测的多样性
    train_preds = final_model.predict(X)
    print(f"训练集预测统计: mean={train_preds.mean():.8f}, std={train_preds.std():.8f}, min={train_preds.min():.8f}, max={train_preds.max():.8f}")
    
    # 打印特征重要性（前10个）
    feature_importance = final_model.feature_importance(importance_type='gain')
    feature_imp_df = pd.DataFrame({
        'feature': dataset.features,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    print(f"\n前10个最重要的特征:")
    print(feature_imp_df.head(10).to_string(index=False))
    
    return final_model

# ============ 4. 信号转换 ============

def convert_ret_to_signal(ret_arr: np.ndarray, params: RetToSignalParameters) -> np.ndarray:
    # 增加一个小的扰动或缩放逻辑，确保不仅输出1.0
    # 这里保持原逻辑，但因为模型预测值会有差异，最终结果会有差异
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

# ============ 5. 执行流程 ============

print("Loading data...")
train_df_raw = load_trainset()
test_df_raw = load_testset()

print("Processing features...")
dataset = split_and_process_dataset(train_df_raw, test_df_raw)

print("Training model...")
model = train_lgbm_model(dataset)
ret_signal_params = RetToSignalParameters(signal_multiplier=SIGNAL_MULTIPLIER)

# ============ 6. API 推理函数 ============

def predict(test: pl.DataFrame, debug: bool = False) -> float:
    """
    推理函数：接收 Polars DataFrame，返回 float 信号。
    使用专门为推理设计的特征工程函数，避免 ewm_mean 在单行数据上的问题
    """
    # 1. 预处理 (必须与训练时一致)
    test = test.rename({'lagged_forward_returns':'target'})
    
    # 2. 使用推理专用的特征工程函数（使用训练时的均值填充空值）
    df = create_example_dataset_inference(test, dataset.feature_means, debug=debug)
    
    # 3. 筛选特征列并转为 Pandas
    # 注意：dataset.features 是我们在训练阶段确定的特征顺序
    try:
        X_test = df.select(dataset.features).to_pandas()
    except Exception as e:
        # 容错：如果测试集缺少某些列（极少见），填充0
        current_cols = df.columns
        missing = [c for c in dataset.features if c not in current_cols]
        for c in missing:
            df = df.with_columns(pl.lit(0.0).alias(c))
        X_test = df.select(dataset.features).to_pandas()

    # 调试：打印标准化前的特征值
    if debug and len(X_test) > 0:
        print(f"标准化前特征值样本 (前3个特征): {X_test.iloc[0, :3].values}")
        print(f"标准化前特征值范围: min={X_test.min().min():.6f}, max={X_test.max().max():.6f}")

    # 4. 标准化特征（参考 template.ipynb，必须与训练时一致）
    X_test_scaled = dataset.scaler.transform(X_test)
    X_test = pd.DataFrame(X_test_scaled, columns=dataset.features, index=X_test.index)
    
    # 调试：打印标准化后的特征值
    if debug and len(X_test) > 0:
        print(f"标准化后特征值样本 (前3个特征): {X_test.iloc[0, :3].values}")
        print(f"标准化后特征值范围: min={X_test.min().min():.6f}, max={X_test.max().max():.6f}")

    # 5. 模型预测
    # LGBM predict 返回的是数组
    raw_pred = model.predict(X_test)[0]
    
    # 调试：打印原始预测值
    if debug:
        print(f"  模型原始预测值 (raw_pred): {raw_pred:.10f}")
    
    # 6. 转换为信号
    final_signal = convert_ret_to_signal(np.array([raw_pred]), ret_signal_params)[0]
    
    # 调试：打印最终信号
    if debug:
        print(f"  最终信号值: {final_signal:.10f}")
    
    return final_signal

# # ============ 7. 启动服务 ============
# print("Starting inference server...")
# inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
#     inference_server.serve()
# else:
#     inference_server.run_local_gateway((str(DATA_PATH),))

Loading data...
Processing features...
使用的特征数量: 13
特征列表: ['S2', 'E2', 'E3', 'P9', 'S1', 'S5', 'I2', 'P8', 'P10', 'P12', 'P13', 'U1', 'U2']
Training model...
开始训练 LGBM...
Fold 1 RMSE: 0.011434
Fold 2 RMSE: 0.013164
Fold 3 RMSE: 0.009316
Fold 4 RMSE: 0.010204
Fold 5 RMSE: 0.010241
使用全量数据重新训练最终模型 (最佳RMSE: 0.009316)...
训练集预测统计: mean=0.00006955, std=0.00175146, min=-0.01959194, max=0.02057918

前10个最重要的特征:
feature  importance
     P8    0.230173
     S2    0.207172
     I2    0.193665
    P12    0.184730
     U1    0.173505
     S5    0.172276
     P9    0.170025
     E3    0.151026
     U2    0.150504
     E2    0.148857


In [6]:
# 生成完整的submission DataFrame
# 获取测试集的所有预测
test_original = pl.read_csv(DATA_PATH / "test.csv")
test_predictions = []

# 调试：检查前两行的原始数据差异
print("调试：检查前两行的原始特征值差异")
row1 = test_original.row(0, named=True)
row2 = test_original.row(1, named=True)
key_cols = ["I1", "I2", "I7", "I9", "M11", "S2", "E2", "E3", "P9"]
for col in key_cols:
    if col in row1 and col in row2:
        val1 = row1[col]
        val2 = row2[col]
        if val1 != val2:
            print(f"  {col}: row1={val1}, row2={val2} (不同)")
        else:
            print(f"  {col}: row1={val2}, row2={val2} (相同)")

for i, row in enumerate(test_original.iter_rows(named=True)):
    # 为每一行创建一个DataFrame
    single_row_df = pl.DataFrame([row])
    # 强制转换类型以匹配训练时
    single_row_df = single_row_df.with_columns(
        pl.exclude("date_id").cast(pl.Float64, strict=False)
    )
    # 使用predict函数生成预测（前两行启用调试）
    pred = predict(single_row_df, debug=(i < 2))
    test_predictions.append(pred)

# 创建submission DataFrame
submission_df = pd.DataFrame({
    'date_id': test_original['date_id'].to_list(),
    'row_id': test_original['date_id'].to_list(),
    'prediction': test_predictions
})

print("Submission DataFrame:")
print(submission_df.head(10))
print(f"\nSubmission shape: {submission_df.shape}")
print(f"\nPrediction statistics:")
print(submission_df['prediction'].describe())

print("-"*50)
# 创建solution DataFrame（包含实际的forward_returns和risk_free_rate）
solution_df = test_original.select([
    'date_id', 
    pl.col('lagged_forward_returns').alias('forward_returns'),
    pl.col('lagged_risk_free_rate').alias('risk_free_rate')
]).to_pandas()

solution_df['row_id'] = solution_df['date_id']

print("Solution DataFrame:")
print(solution_df.head())
print(f"\nForward returns 统计:")
print(solution_df['forward_returns'].describe())
# 导入SharpeRatio评分函数并计算得分
from SharpeRatio import score

try:
    sharpe_score = score(
        solution=solution_df, 
        submission=submission_df, 
        row_id_column_name='date_id'
    )
    print(f"{'='*60}")
    print(f"✓ Sharpe Ratio Score: {sharpe_score:.6f}")
    print(f"{'='*60}")
    
    # 显示策略详情
    print(f"\n策略表现:")
    print(f"  - 平均预测信号: {submission_df['prediction'].mean():.4f}")
    print(f"  - 信号标准差: {submission_df['prediction'].std():.4f}")
    print(f"  - 信号范围: [{submission_df['prediction'].min():.4f}, {submission_df['prediction'].max():.4f}]")
    
except Exception as e:
    print(f"❌ 计算Sharpe Ratio时出错: {e}")
    import traceback
    traceback.print_exc()


调试：检查前两行的原始特征值差异
  I1: row1=0.306216931216931, row2=0.305886243386243 (不同)
  I2: row1=1.02575637127463, row2=0.989571475584055 (不同)
  I7: row1=0.256283068783069, row2=0.255952380952381 (不同)
  I9: row1=0.676060977652112, row2=0.660556174585453 (不同)
  M11: row1=-0.236099720197407, row2=-0.220753939458671 (不同)
  S2: row1=0.0857167792140139, row2=0.28169041152573 (不同)
  E2: row1=1.25906537584674, row2=1.19346789571564 (不同)
  E3: row1=1.71258010208486, row2=1.6400541928101 (不同)
  P9: row1=0.0297619047619048, row2=0.0337301587301587 (不同)
  原始基础列值:
    I1 = 0.306216931216931
    I2 = 1.02575637127463
    I7 = 0.256283068783069
    I9 = 0.676060977652112
    M11 = -0.236099720197407
  填充后基础列值:
    I1 = 0.306216931216931
    I2 = 1.02575637127463
    I7 = 0.256283068783069
    I9 = 0.676060977652112
    M11 = -0.236099720197407
  计算出的 U1 = 0.7195394400576991, U2 = -0.36172769452182374
  最终特征值 (前5个):
    S2 = 0.0857167792140139
    E2 = 1.25906537584674
    E3 = 1.71258010208486
    P9 = 0.02976